In [ ]:
import requests
import pandas as pd
import matplotlib.pyplot as plt
from datetime import datetime
from pathlib import Path

pd.set_option("display.float_format", lambda x: f"{x:,.4f}")
pd.set_option("display.max_columns", None)
pd.set_option("display.width", 200)

API = "https://api.hyperliquid.xyz/info"
ADDR = "0x7532dA34edA8f3e383bC480d50e7A7d3b12c07f2"   # <-- put any wallet address here

def post(payload):
    r = requests.post(API, json=payload, timeout=15)
    r.raise_for_status()
    return r.json()

def spot_balance_usd():
    spot = post({"type": "spotClearinghouseState", "user": ADDR})
    hype_mids = post({"type": "allMids", "dex": ""})
    hype_price = float(hype_mids.get("HYPE") or 0)
    if hype_price <= 0:
        raise ValueError("Could not retrieve a valid HYPE mid-price.")

    total_usd = 0.0
    priced = []
    excluded = []
    for balance in spot.get("balances", []):
        amount = float(balance.get("total", 0) or 0)
        if not amount:
            continue
        coin = balance.get("coin", "")
        if coin == "USDC":
            price = 1.0
        elif coin == "HYPE":
            price = hype_price
        else:
            excluded.append((coin, amount))
            continue
        value_usd = amount * price
        total_usd += value_usd
        priced.append((coin, amount, price, value_usd))
    return total_usd, priced, excluded


In [ ]:
CSV_PATH = Path("hyperliquid_account_history.csv")
FETCH_FULL_HISTORY = False
if CSV_PATH.exists():
    saved = pd.read_csv(CSV_PATH, parse_dates=["ts"])
    saved["ts"] = pd.to_datetime(saved["ts"], utc=True).dt.tz_localize(None)
else:
    saved = pd.DataFrame(columns=["ts", "accountValue", "source", "balance", "open_position_value", "unrealized_pnl", "realized_pnl"])

history = []
if saved.empty or FETCH_FULL_HISTORY:
    portfolio = post({"type": "portfolio", "user": ADDR})
    portfolio_by_period = {item[0]: item[1] for item in portfolio if isinstance(item, (list, tuple)) and len(item) == 2 and isinstance(item[1], dict)}
    window = portfolio_by_period.get("allTime")
    if not window or not window.get("accountValueHistory"):
        raise ValueError("No allTime account value history was returned.")
    history = window["accountValueHistory"]

perp_dexes = post({"type": "perpDexs"})
dex_names = [""] + [dex["name"] for dex in perp_dexes if dex and dex.get("name")]
current_states = {dex: post({"type": "clearinghouseState", "user": ADDR, "dex": dex}) for dex in dex_names}
balance_total, priced_spot, unpriced_spot = spot_balance_usd()
unrealized_pnl_total = sum(float(p.get("position", {}).get("unrealizedPnl", 0) or 0) for a in current_states.values() for p in a.get("assetPositions", []))
position_value_total = sum(abs(float(p.get("position", {}).get("positionValue", 0) or 0)) for a in current_states.values() for p in a.get("assetPositions", []))
current_account_value = balance_total + unrealized_pnl_total

# Fetch fills once; cell 3 uses this same data for spot/perp realized PnL stats.
realized_fills = []
end_ms = int(pd.Timestamp.now(tz="UTC").timestamp() * 1000)
for _ in range(5):
    batch = post({"type": "userFillsByTime", "user": ADDR, "startTime": 0, "endTime": end_ms, "aggregateByTime": True})
    if not isinstance(batch, list) or not batch:
        break
    realized_fills.extend(batch)
    valid_times = [int(f["time"]) for f in batch if f.get("time") is not None]
    if len(batch) < 2000 or not valid_times or len(realized_fills) >= 10000:
        break
    next_end_ms = min(valid_times) - 1
    if next_end_ms >= end_ms:
        break
    end_ms = next_end_ms
realized_rows = [{"time": int(f["time"]), "coin": f.get("coin", ""),
                  "assetType": "spot" if str(f.get("coin", "")).startswith("@") else "perp",
                  "dir": f.get("dir", ""),
                  "notional": abs(float(f.get("px", 0) or 0) * float(f.get("sz", 0) or 0)),
                  "closedPnl": float(f.get("closedPnl", 0) or 0)}
                 for f in realized_fills if f.get("time") is not None]
realized_df = pd.DataFrame(realized_rows)
if not realized_df.empty:
    realized_df["ts"] = pd.to_datetime(realized_df["time"], unit="ms", utc=True)
    realized_df["day"] = realized_df["ts"].dt.floor("D").dt.tz_localize(None)

new_history = pd.DataFrame(history, columns=["ts", "accountValue"])
new_history["ts"] = pd.to_datetime(new_history["ts"], unit="ms")
new_history["accountValue"] = new_history["accountValue"].astype(float)
new_history["source"] = "portfolio_allTime"
for col in ["balance", "open_position_value", "unrealized_pnl", "realized_pnl"]:
    new_history[col] = float("nan")
if not saved.empty:
    saved_history = saved[saved["source"] == "portfolio_allTime"]
    last_history_ts = saved_history["ts"].max() if not saved_history.empty else pd.Timestamp.min
    new_history = new_history[new_history["ts"] > last_history_ts]
else:
    saved_history = saved
history_rows_added = len(new_history)

live_snapshot = pd.DataFrame([{
    "ts": pd.Timestamp.now(tz="UTC").tz_localize(None), "accountValue": current_account_value,
    "source": "live_snapshot", "balance": balance_total, "open_position_value": position_value_total,
    "unrealized_pnl": unrealized_pnl_total, "realized_pnl": float("nan"),
}])
if not realized_df.empty:
    daily_realized_new = realized_df[realized_df["assetType"] == "perp"].groupby("day", as_index=False)["closedPnl"].sum()
    daily_realized_new = daily_realized_new.rename(columns={"day": "ts", "closedPnl": "realized_pnl"})
    daily_realized_new["source"] = "perp_realized_daily"
    for col in ["accountValue", "balance", "open_position_value", "unrealized_pnl"]:
        daily_realized_new[col] = float("nan")
else:
    daily_realized_new = pd.DataFrame(columns=["ts", "accountValue", "source", "balance", "open_position_value", "unrealized_pnl", "realized_pnl"])

columns = ["ts", "accountValue", "source", "balance", "open_position_value", "unrealized_pnl", "realized_pnl"]
chart_df = pd.concat([saved.reindex(columns=columns), new_history.reindex(columns=columns), live_snapshot.reindex(columns=columns), daily_realized_new.reindex(columns=columns)], ignore_index=True)
chart_df["ts"] = pd.to_datetime(chart_df["ts"])
chart_df = chart_df.drop_duplicates(subset=["ts", "source"], keep="last").sort_values("ts").reset_index(drop=True)
chart_df.to_csv(CSV_PATH, index=False)

# Keep API account value and daily perp PnL as separate chart panels.
account_series = chart_df.loc[chart_df["source"] == "portfolio_allTime"].set_index("ts")["accountValue"].dropna()
daily_account = account_series.resample("1D").last().dropna()
realized_series = chart_df.loc[chart_df["source"] == "perp_realized_daily"].set_index("ts")["realized_pnl"].dropna()
realized_series = realized_series.groupby(level=0).last().sort_index()
unrealized_series = chart_df.loc[chart_df["source"] == "live_snapshot"].set_index("ts")["unrealized_pnl"].dropna()
daily_unrealized = unrealized_series.resample("1D").last().dropna()
if not realized_series.empty or not daily_unrealized.empty:
    date_min = min(x.index.min() for x in [realized_series, daily_unrealized] if not x.empty)
    date_max = max(x.index.max() for x in [realized_series, daily_unrealized] if not x.empty)
    day_index = pd.date_range(date_min, date_max, freq="1D")
    daily_realized = realized_series.reindex(day_index, fill_value=0.0)
    daily_unrealized = daily_unrealized.reindex(day_index)
else:
    day_index = pd.DatetimeIndex([])
    daily_realized = pd.Series(dtype=float, index=day_index)
    daily_unrealized = pd.Series(dtype=float, index=day_index)

realized_avg_day = daily_realized.mean() if not daily_realized.empty else float("nan")
u_change = daily_unrealized.dropna().diff().dropna()
u_avg_day = u_change.mean() if not u_change.empty else float("nan")
net_change = daily_realized.add(daily_unrealized.ffill().diff().fillna(0), fill_value=0).dropna()
net_avg_day = net_change.mean() if not net_change.empty else float("nan")
equity_for_returns = daily_account.reindex(net_change.index).ffill()
daily_return_pct = (net_change / equity_for_returns.shift(1) * 100).replace([float("inf"), -float("inf")], float("nan")).dropna()
avg_daily_return_pct = daily_return_pct.mean() if not daily_return_pct.empty else float("nan")
max_daily_gain_pct = daily_return_pct.max() if not daily_return_pct.empty else float("nan")
max_daily_loss_pct = daily_return_pct.min() if not daily_return_pct.empty else float("nan")
cumulative_pnl = daily_realized.cumsum() + daily_unrealized.ffill().fillna(0)
if not cumulative_pnl.empty:
    pnl_dd_usd = (cumulative_pnl - cumulative_pnl.cummax()).min()
else:
    pnl_dd_usd = float("nan")

if not daily_account.empty:
    account_peak = daily_account.cummax()
    account_dd_usd = (daily_account - account_peak).min()
    positive = daily_account > 0
    account_dd_pct = ((daily_account[positive] / account_peak[positive]) - 1).min() * 100 if positive.any() else float("nan")
else:
    account_dd_usd = account_dd_pct = float("nan")

# BTC daily close returns are the benchmark for PnL alpha, beta and Sharpe.
alpha_annual = beta = sharpe_annual = float("nan")
btc_sample_n = 0
try:
    if len(day_index) > 1:
        btc_start_ms = int(day_index.min().timestamp() * 1000)
        btc_end_ms = int(pd.Timestamp.now(tz="UTC").timestamp() * 1000)
        btc_candles = post({
            "type": "candleSnapshot",
            "req": {"coin": "BTC", "interval": "1d", "startTime": btc_start_ms, "endTime": btc_end_ms},
        })
        btc_df = pd.DataFrame(btc_candles)
        if not btc_df.empty:
            btc_df["day"] = pd.to_datetime(btc_df["t"], unit="ms", utc=True).dt.tz_localize(None).dt.normalize()
            btc_close = btc_df.assign(close=btc_df["c"].astype(float)).set_index("day")["close"].sort_index()
            btc_returns = btc_close.pct_change()
            equity = daily_account.reindex(net_change.index).ffill()
            portfolio_returns = net_change / equity.shift(1)
            compare = pd.concat([portfolio_returns.rename("pnl"), btc_returns.reindex(portfolio_returns.index).rename("btc")], axis=1).dropna()
            btc_sample_n = len(compare)
            if btc_sample_n >= 2 and compare["btc"].var() > 0:
                beta = compare["pnl"].cov(compare["btc"]) / compare["btc"].var()
                alpha_annual = (compare["pnl"].mean() - beta * compare["btc"].mean()) * 365 * 100
                pnl_std = compare["pnl"].std()
                sharpe_annual = compare["pnl"].mean() / pnl_std * (365 ** 0.5) if pnl_std else float("nan")
except Exception as exc:
    print(f"BTC metrics unavailable: {type(exc).__name__}")

print("=== PNL / ACCT ===")
print(f"Acct value: ${daily_account.iloc[-1]:,.2f}" if not daily_account.empty else "Acct value: n/a")
print(f"1D: {daily_account.pct_change().iloc[-1] * 100:+.2f}%" if len(daily_account) > 1 else "1D: need 2 daily values")
print(f"Acct Max DD: ${account_dd_usd:,.2f} ({account_dd_pct:.2f}%)" if pd.notna(account_dd_usd) else "Acct Max DD: n/a")
print(f"PnL Max DD: ${pnl_dd_usd:,.2f}" if pd.notna(pnl_dd_usd) else "PnL Max DD: n/a")
realized_today = daily_realized.iloc[-1] if not daily_realized.empty else 0.0
u_change_today = daily_unrealized.dropna().diff().iloc[-1] if len(daily_unrealized.dropna()) > 1 else float("nan")
print(f"Realized 1D: ${realized_today:+,.2f} | Avg/d: ${realized_avg_day:+,.2f}" if pd.notna(realized_avg_day) else f"Realized 1D: ${realized_today:+,.2f} | Avg/d: n/a")
print(f"uPnL: ${unrealized_pnl_total:,.2f} | 1D Δ: ${u_change_today:+,.2f} | Avg/d Δ: ${u_avg_day:+,.2f}" if pd.notna(u_avg_day) and pd.notna(u_change_today) else f"uPnL: ${unrealized_pnl_total:,.2f} | Avg/d Δ: n/a")
print(f"Avg daily change: ${net_avg_day:+,.2f} ({avg_daily_return_pct:+.2f}%) | Max daily gain: {max_daily_gain_pct:+.2f}% | Max daily loss: {max_daily_loss_pct:+.2f}%" if pd.notna(avg_daily_return_pct) else "Daily return stats: need matched daily PnL and account values")
print(f"BTC α: {alpha_annual:+.2f}%/yr | β: {beta:.2f} | Sharpe: {sharpe_annual:.2f} | n={btc_sample_n}" if pd.notna(beta) and pd.notna(sharpe_annual) else f"BTC α / β / Sharpe: need more matched daily data (n={btc_sample_n})")
print(f"CSV: {CSV_PATH.resolve()} | New API rows: {history_rows_added}")
if len(realized_fills) >= 10000:
    print("Realized history limited to latest 10,000 fills.")

fig = plt.figure(figsize=(12, 14))
grid = fig.add_gridspec(4, 1)
ax_acct = fig.add_subplot(grid[0, 0])
ax_pnl = fig.add_subplot(grid[1, 0], sharex=ax_acct)
ax_cum = fig.add_subplot(grid[2, 0], sharex=ax_acct)
ax_hist = fig.add_subplot(grid[3, 0])
if not daily_account.empty:
    ax_acct.plot(daily_account.index, daily_account.values, linewidth=2, color="#6c757d", label="Account Value")
    ax_acct.legend(loc="upper left", bbox_to_anchor=(1.01, 1), borderaxespad=0)
ax_acct.grid(alpha=0.3)
if not daily_realized.empty:
    ax_pnl.plot(day_index, daily_realized.values, linewidth=2, marker="o", markersize=3, color="#457b9d", label="Realized")
if not daily_unrealized.empty:
    ax_pnl.plot(day_index, daily_unrealized.values, linewidth=2, marker="o", markersize=3, color="#e76f51", label="Unrealized")
ax_pnl.axhline(0, color="#777", linewidth=0.8)
ax_pnl.grid(alpha=0.3)
if not daily_realized.empty or not daily_unrealized.empty:
    ax_pnl.legend(loc="upper left", bbox_to_anchor=(1.01, 1), borderaxespad=0)
if not cumulative_pnl.empty:
    ax_cum.plot(cumulative_pnl.index, cumulative_pnl.values, linewidth=2, color="#8e44ad", label="Cumulative PnL")
    ax_cum.legend(loc="upper left", bbox_to_anchor=(1.01, 1), borderaxespad=0)
ax_cum.axhline(0, color="#777", linewidth=0.8)
ax_cum.grid(alpha=0.3)
if not daily_return_pct.empty:
    ax_hist.hist(daily_return_pct.values, bins="auto", color="#457b9d", alpha=0.8, label="Daily returns")
    ax_hist.axvline(avg_daily_return_pct, color="#e76f51", linestyle="--", linewidth=2, label="Average")
    ax_hist.axvline(0, color="#555", linewidth=0.8)
    ax_hist.legend(loc="upper left", bbox_to_anchor=(1.01, 1), borderaxespad=0)
else:
    ax_hist.text(0.5, 0.5, "Need matched daily PnL and account values", ha="center", va="center", transform=ax_hist.transAxes)
ax_hist.set_title("Daily Return Distribution")
ax_hist.set_xlabel("Gain / loss (%)")
ax_hist.set_ylabel("Days")
ax_hist.grid(alpha=0.3)
fig.tight_layout(rect=[0, 0, 0.82, 1])
plt.show()


In [ ]:
# The default clearinghouseState only covers the main perp DEX. Load every listed HIP-3 DEX too.
perp_dexes = post({"type": "perpDexs"})
dex_names = [""] + [dex["name"] for dex in perp_dexes if dex and dex.get("name")]
states = {dex: post({"type": "clearinghouseState", "user": ADDR, "dex": dex}) for dex in dex_names}
state = states[""]  # Keep the account summary for the main Hyperliquid perp DEX.
positions = [(dex or "Hyperliquid", p) for dex, dex_state in states.items() for p in dex_state.get("assetPositions", [])]

rows = []
for dex_name, p in positions:
    pos = p["position"]
    lev = pos.get("leverage") or {}
    funding = pos.get("cumFunding") or {}
    number = lambda value: float(value) if value not in (None, "") else 0.0
    size = number(pos.get("szi"))
    position_value = number(pos.get("positionValue"))
    mark_price = abs(position_value) / abs(size) if size else 0
    liquidation_price = number(pos.get("liquidationPx"))
    if liquidation_price > 0 and mark_price > 0:
        liq_distance_pct = ((mark_price - liquidation_price) / mark_price * 100
                           if size > 0 else (liquidation_price - mark_price) / mark_price * 100)
    else:
        liq_distance_pct = float("nan")
    # Hyperliquid's cumulative position field is opposite the account UI cash-flow sign.
    # Negative means this position paid funding; positive means it received funding.
    funding_since_open = -number(funding.get("sinceOpen"))
    margin = number(pos.get("marginUsed"))
    rows.append({
        "Coin": pos["coin"], "Sz": size, "Side": "LONG" if size > 0 else "SHORT",
        "Entry": number(pos.get("entryPx")), "Mark": mark_price, "Pos": position_value,
        "Liq%": liq_distance_pct, "uPnL": number(pos.get("unrealizedPnl")),
        "ROE": number(pos.get("returnOnEquity")) * 100,
        "Lev.": abs(position_value) / margin if margin else float("nan"),
        "Liq Px": liquidation_price, "Margin": margin, "Funding": funding_since_open,
    })

if rows:
    df_pos = pd.DataFrame(rows).sort_values(
        "Pos", key=lambda values: values.abs(), ascending=False
    ).reset_index(drop=True)
    usd_format = lambda value: f"-${abs(value):,.2f}" if value < 0 else f"${value:,.2f}"
    # Color PnL/funding and flag positions that are close to liquidation.
    styler = df_pos.style
    color_map = getattr(styler, "map", None)
    if color_map is None:
        color_map = styler.applymap
    styled = color_map(
        lambda v: "color: #2a9d8f" if isinstance(v, (int, float)) and v > 0
        else ("color: #e63946" if isinstance(v, (int, float)) and v < 0 else ""),
        subset=["uPnL", "ROE", "Funding"],
    )
    styled = styled.map(
        lambda v: "" if pd.isna(v) else ("color: #e63946" if v <= 10
        else ("color: #d97706" if v <= 25 else "color: #2a9d8f")),
        subset=["Liq%"],
    ) if hasattr(styled, "map") else styled.applymap(
        lambda v: "" if pd.isna(v) else ("color: #e63946" if v <= 10
        else ("color: #d97706" if v <= 25 else "color: #2a9d8f")),
        subset=["Liq%"],
    )
    styled = styled.format({
        "Sz": "{:,.2f}", "Entry": "{:,.2f}", "Mark": "{:,.2f}",
        "Pos": usd_format, "Liq%": "{:,.2f}%", "uPnL": usd_format,
        "ROE": "{:,.2f}%", "Lev.": "{:,.2f}x", "Liq Px": "{:,.2f}",
        "Margin": usd_format, "Funding": usd_format,
    })
    display(styled)
else:
    print("No open positions.")

# Totals across all open perp positions and DEXes.
position_value_total = sum(abs(row["Pos"]) for row in rows)
unrealized_pnl_total = sum(row["uPnL"] for row in rows)
margin_used_total = sum(row["Margin"] for row in rows)
funding_total = sum(row["Funding"] for row in rows)
balance_total, priced_spot, unpriced_spot = spot_balance_usd()
current_account_value = balance_total + unrealized_pnl_total
overall_roe = unrealized_pnl_total / margin_used_total * 100 if margin_used_total else 0
total_leverage = position_value_total / margin_used_total if margin_used_total else float("nan")
margin_pct_balance = margin_used_total / balance_total * 100 if balance_total else float("nan")

print("\n=== POSITIONS ===")
print(f"Pos: ${position_value_total:,.2f}")
print(f"uPnL: ${unrealized_pnl_total:,.2f}")
print(f"ROE: {overall_roe:,.2f}%")
print(f"Margin: ${margin_used_total:,.2f}")
print(f"Margin % bal: {margin_pct_balance:,.2f}%" if pd.notna(margin_pct_balance) else "Margin % bal: n/a")
print(f"Lev. (Pos / Margin): {total_leverage:,.2f}x" if pd.notna(total_leverage) else "Lev.: n/a")
print(f"Funding: ${funding_total:,.2f}")

# Realized PnL stats, split by spot and perp fills fetched in cell 2.
now_utc = pd.Timestamp.now(tz="UTC")
q_start_month = ((now_utc.month - 1) // 3) * 3 + 1
q_start = pd.Timestamp(year=now_utc.year, month=q_start_month, day=1, tz="UTC")
y_start = pd.Timestamp(year=now_utc.year, month=1, day=1, tz="UTC")
perp_df = realized_df[realized_df["assetType"] == "perp"].copy() if not realized_df.empty else pd.DataFrame()
spot_df = realized_df[realized_df["assetType"] == "spot"].copy() if not realized_df.empty else pd.DataFrame()

def realized_pnl_stats(frame):
    if frame.empty:
        return {"total": 0.0, "1d": 0.0, "7d": 0.0, "30d": 0.0, "qtd": 0.0, "ytd": 0.0,
                "wins": 0, "losses": 0, "win_p": float("nan"), "loss_p": float("nan"),
                "avg_win": float("nan"), "avg_loss": float("nan"), "avg_win_pct": float("nan"), "avg_loss_pct": float("nan"), "ev": float("nan"), "fills": 0}
    win_rows = frame.loc[frame["closedPnl"] > 0]
    loss_rows = frame.loc[frame["closedPnl"] < 0]
    wins = win_rows["closedPnl"]
    losses = loss_rows["closedPnl"]
    win_returns = (wins / win_rows["notional"] * 100).replace([float("inf"), -float("inf")], float("nan")).dropna()
    loss_returns = (losses / loss_rows["notional"] * 100).replace([float("inf"), -float("inf")], float("nan")).dropna()
    closed_count = len(wins) + len(losses)
    win_p = len(wins) / closed_count if closed_count else float("nan")
    loss_p = len(losses) / closed_count if closed_count else float("nan")
    avg_win = wins.mean() if not wins.empty else 0.0
    avg_loss = losses.mean() if not losses.empty else 0.0
    ev = win_p * avg_win + loss_p * avg_loss if closed_count else float("nan")
    return {
        "total": frame["closedPnl"].sum(),
        "1d": frame.loc[frame["ts"] >= now_utc - pd.Timedelta(days=1), "closedPnl"].sum(),
        "7d": frame.loc[frame["ts"] >= now_utc - pd.Timedelta(days=7), "closedPnl"].sum(),
        "30d": frame.loc[frame["ts"] >= now_utc - pd.Timedelta(days=30), "closedPnl"].sum(),
        "qtd": frame.loc[frame["ts"] >= q_start, "closedPnl"].sum(),
        "ytd": frame.loc[frame["ts"] >= y_start, "closedPnl"].sum(),
        "wins": len(wins), "losses": len(losses), "win_p": win_p, "loss_p": loss_p,
        "avg_win": avg_win, "avg_loss": avg_loss,
        "avg_win_pct": win_returns.mean() if not win_returns.empty else float("nan"),
        "avg_loss_pct": loss_returns.mean() if not loss_returns.empty else float("nan"),
        "ev": ev, "fills": len(frame),
    }

for frame, title in [(spot_df, "SPOT"), (perp_df, "PERP")]:
    stats = realized_pnl_stats(frame)
    print(f"\n=== REALIZED {title} ===")
    print(f"PnL: ${stats['total']:,.2f} | 1D: ${stats['1d']:,.2f} | QTD: ${stats['qtd']:,.2f} | YTD: ${stats['ytd']:,.2f} | Since inc.: ${stats['total']:,.2f}")
    print(f"W: {stats['wins']} ({stats['win_p']:.1%}) | Avg W: ${stats['avg_win']:,.2f} ({stats['avg_win_pct']:+.2f}% ntl)")
    print(f"L: {stats['losses']} ({stats['loss_p']:.1%}) | Avg L: ${stats['avg_loss']:,.2f} ({stats['avg_loss_pct']:+.2f}% ntl)")
    print(f"EV/close: ${stats['ev']:,.2f}" if pd.notna(stats['ev']) else "EV/close: n/a")

if not perp_df.empty:
    dir_lower = perp_df["dir"].fillna("").astype(str).str.lower()
    close_mask = dir_lower.str.contains("close|liquidat") | (perp_df["closedPnl"] != 0)
    close_rows = perp_df.loc[close_mask].drop_duplicates(subset=["time", "coin", "dir"])
    liq_rows = perp_df.loc[dir_lower.str.contains("liquidat")].drop_duplicates(subset=["time", "coin", "dir"])
    liquidation_count = len(liq_rows)
    close_count = len(close_rows)
    liquidation_pct = liquidation_count / close_count * 100 if close_count else 0.0
    perp_realized_total = perp_df["closedPnl"].sum()
    liquidation_pnl = liq_rows["closedPnl"].sum()
    liquidation_pnl_pct = liquidation_pnl / perp_realized_total * 100 if perp_realized_total else float("nan")
else:
    liquidation_count = close_count = 0
    liquidation_pct = liquidation_pnl = 0.0
    liquidation_pnl_pct = float("nan")
print(f"Liq: {liquidation_count}/{close_count} closes ({liquidation_pct:.2f}%) | Liq PnL: ${liquidation_pnl:,.2f} ({liquidation_pnl_pct:.2f}% of perp PnL)" if pd.notna(liquidation_pnl_pct) else f"Liq: {liquidation_count}/{close_count} closes ({liquidation_pct:.2f}%) | Liq PnL: ${liquidation_pnl:,.2f} (share n/a)")
print("Win/loss and liquidation counts use available fill history (API cap: 10,000).")

# Portfolio API value remains a separate reference series.
portfolio = post({"type": "portfolio", "user": ADDR})
portfolio_by_period = {item[0]: item[1] for item in portfolio if isinstance(item, (list, tuple)) and len(item) == 2}
all_time_history = portfolio_by_period.get("allTime", {}).get("accountValueHistory", [])
portfolio_latest = float(all_time_history[-1][1]) if all_time_history else float("nan")
